<a href="https://colab.research.google.com/github/catalinaMaria1/TemaIA/blob/Tema3%C3%8EA/FP_Growth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import warnings
warnings.filterwarnings("ignore")


In [7]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

df = pd.read_csv("consumption_user.csv", low_memory=False)

df = df[["RESPONDER", "ROUND", "FOOD_TYPE"]].dropna()

df["FOOD_TYPE"] = (
    df["FOOD_TYPE"]
    .astype(str)
    .str.strip()
    .str.lower()
)

food_group_map = {
    "1": "cereals",
    "2": "vegetables",
    "3": "meat_fish",
    "4": "dairy",
    "5": "fats_oils",
    "6": "sugars",
    "7": "fruits",
    "8": "legumes",
    "9": "beverages"
}

df["FOOD_GROUP"] = df["FOOD_TYPE"].map(food_group_map)

df = df.dropna(subset=["FOOD_GROUP"])

df["TRANSACTION_ID"] = (
    df["RESPONDER"].astype(str) + "_" +
    df["ROUND"].astype(str)
)

transactions = (
    df.groupby("TRANSACTION_ID")["FOOD_GROUP"]
    .apply(lambda x: list(set(x)))
)

transactions = transactions[transactions.apply(len) >= 2]

print("Tranzacții valide:", len(transactions))

te = TransactionEncoder()
df_bin = pd.DataFrame(
    te.fit(transactions).transform(transactions),
    columns=te.columns_
)

frequent_itemsets = fpgrowth(
    df_bin,
    min_support=0.2,
    use_colnames=True
)

print("Itemset-uri frecvente (FP-Growth):", frequent_itemsets.shape[0])

if frequent_itemsets.shape[0] > 0:
    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=0.6
    )

    rules = rules.sort_values(by=["lift", "confidence"], ascending=False)

    print("\nFP-GROWTH – REGULI FINALE")
    print(rules[["antecedents", "consequents", "support", "confidence", "lift"]])
else:
    print("Nu există reguli – date insuficiente.")


Tranzacții valide: 2
Itemset-uri frecvente (FP-Growth): 15

FP-GROWTH – REGULI FINALE
                         antecedents                       consequents  \
0                        (meat_fish)                      (vegetables)   
1                       (vegetables)                       (meat_fish)   
2                       (vegetables)                           (dairy)   
3                            (dairy)                      (vegetables)   
4                          (cereals)                      (vegetables)   
5                       (vegetables)                         (cereals)   
6                        (meat_fish)                           (dairy)   
7                            (dairy)                       (meat_fish)   
8                          (cereals)                       (meat_fish)   
9                        (meat_fish)                         (cereals)   
10                         (cereals)                           (dairy)   
11                        